In [6]:
import io
import re
import sys
import warnings
import requests
import joblib  # <--- NEW: For saving the model
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score # <--- NEW: Regression metrics
from xgboost import XGBRegressor # <--- NEW: The Regressor

warnings.filterwarnings("ignore")

UK_GPG_PAGE = "https://gender-pay-gap.service.gov.uk/Viewing/download"

CSV_SIGNATURE_COLUMNS = {
    "EmployerName",
    "DiffMedianHourlyPercent",
    "EmployerSize",
    "SicCodes",
    "MaleTopQuartile", "FemaleTopQuartile",
    "MaleUpperMiddleQuartile", "FemaleUpperMiddleQuartile",
    "MaleLowerMiddleQuartile", "FemaleLowerMiddleQuartile",
    "MaleLowerQuartile", "FemaleLowerQuartile"
}

# ... [Keep your existing looks_like_csv, fetch_candidate_links, try_fetch_csv_from_links functions exactly as they are] ...
# (I am omitting them here to save space, but you should keep them in your file)

def looks_like_csv(text: str) -> bool:
    """Heuristic: CSV if first line contains comma-separated headers and at least one known column."""
    first_line = text.splitlines()[0] if text else ""
    if "," not in first_line:
        return False
    headers = [h.strip() for h in first_line.split(",")]
    return any(col in headers for col in CSV_SIGNATURE_COLUMNS)

def fetch_candidate_links(page_url: str) -> list:
    """Fetch the GPG download page and extract all <a href> links (absolute URLs)."""
    resp = requests.get(page_url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    hrefs = []
    for a in soup.select("a[href]"):
        href = a.get("href")
        if href:
            hrefs.append(urljoin(page_url, href))
    # Include the page itself as a fallback (some sites serve CSV via query params)
    hrefs = list(dict.fromkeys(hrefs))  # unique, preserve order
    return hrefs

def try_fetch_csv_from_links(links: list) -> pd.DataFrame:
    """Iterate links, request each; if HTTP 200 and content looks like CSV, parse and return DataFrame."""
    for link in links:
        try:
            r = requests.get(link, timeout=30)
            if r.status_code != 200:
                continue
            # Check header and content-type hints
            ctype = r.headers.get("Content-Type", "").lower()
            if "text/csv" in ctype or "application/csv" in ctype or looks_like_csv(r.text):
                df = pd.read_csv(io.StringIO(r.text))
                # Basic schema check
                if "DiffMedianHourlyPercent" in df.columns:
                    print(f"Using CSV: {link}")
                    return df
        except Exception:
            # Try next link
            continue
    raise RuntimeError("Could not find a valid CSV from the GPG download page links.")

def prepare_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    UPDATED: Keeps the raw 'DiffMedianHourlyPercent' for regression.
    Removes extreme outliers to prevent skewing the model.
    """
    df = df.copy()
    if "DiffMedianHourlyPercent" not in df.columns:
        raise ValueError("CSV does not contain 'DiffMedianHourlyPercent'.")

    # 1. Convert Target to Numeric (Force errors to NaN)
    df["target"] = pd.to_numeric(df["DiffMedianHourlyPercent"], errors="coerce")

    # 2. Drop Missing Targets
    df = df.dropna(subset=["target"])

    # 3. Remove Outliers (e.g., Gaps > 100% or < -100% are usually data errors)
    df = df[(df["target"] > -100) & (df["target"] < 100)]

    feature_cols = [
        "EmployerSize", "SicCodes",
        "MaleTopQuartile", "FemaleTopQuartile",
        "MaleUpperMiddleQuartile", "FemaleUpperMiddleQuartile",
        "MaleLowerMiddleQuartile", "FemaleLowerMiddleQuartile",
        "MaleLowerQuartile", "FemaleLowerQuartile"
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]

    # Clean Numeric Features
    quartiles = [c for c in feature_cols if "Quartile" in c]
    for c in quartiles:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

    # Clean Categorical Features
    if "EmployerSize" in df.columns:
        df["EmployerSize"] = df["EmployerSize"].fillna("unknown")
    if "SicCodes" in df.columns:
        # Take the first SIC code if multiple exist
        df["SicCodes"] = df["SicCodes"].astype(str).str.split(",").str[0].str.strip()

    # Return Features + Target
    return df[feature_cols + ["target"]]

def build_pipeline(cat_cols, num_cols):
    """
    UPDATED: Uses XGBRegressor instead of LogisticRegression.
    """
    transformers = []
    if cat_cols:
        # handle_unknown='ignore' is crucial so new industries in production don't crash the code
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols))

    ct = ColumnTransformer(transformers=transformers, remainder="passthrough")

    # XGBoost Regressor Configuration
    xgb = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        objective='reg:squarederror', # Standard regression objective
        random_state=42,
        n_jobs=-1
    )

    pipe = Pipeline(steps=[("prep", ct), ("model", xgb)])
    return pipe

def main():
    # ... [Keep your fetching logic here] ...
    # assuming you already have 'df_raw' from your fetch function:

    print("Fetching the UK GPG download page…")
    links = fetch_candidate_links(UK_GPG_PAGE)
    print(f"Found {len(links)} candidate links. Probing for CSV…")
    df_raw = try_fetch_csv_from_links(links)
    print("Raw rows:", len(df_raw))

    print("Preparing dataset for REGRESSION...")
    df = prepare_dataset(df_raw)
    print(f"Rows after cleaning: {len(df)}")

    X = df.drop(columns=["target"])
    y = df["target"].values

    cat_cols = [c for c in X.columns if X[c].dtype == object]
    num_cols = [c for c in X.columns if c not in cat_cols]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print("Training XGBoost Regressor...")
    model = build_pipeline(cat_cols, num_cols)
    model.fit(X_train, y_train)

    # EVALUATION
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n=== Regression Results ===")
    print(f"Mean Absolute Error (MAE): {mae:.2f}%")
    print(f"(On average, the prediction is off by {mae:.2f} percentage points)")
    print(f"R2 Score: {r2:.3f}")

    # SAVING THE MODEL
    print("\nSaving model to 'bias_predictor_uk.pkl'...")
    joblib.dump(model, 'bias_predictor_uk.pkl')
    print("Done.")

if __name__ == "__main__":
    # Ensure you have the 'df_raw' logic from your previous script here
    # or just call main() if integrating into the full file.
    main()

Fetching the UK GPG download page…
Found 25 candidate links. Probing for CSV…
Using CSV: https://gender-pay-gap.service.gov.uk/viewing/download-data/2025
Raw rows: 1007
Preparing dataset for REGRESSION...
Rows after cleaning: 1006
Training XGBoost Regressor...

=== Regression Results ===
Mean Absolute Error (MAE): 6.65%
(On average, the prediction is off by 6.65 percentage points)
R2 Score: 0.557

Saving model to 'bias_predictor_uk.pkl'...
Done.
